[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch05/NN_DL_ch05_LSTMViz/NN_DL_ch05_LSTMViz.ipynb)

# NN_DL_ch05_LSTMViz

**LSTM Architecture: Gate Diagrams & Vanishing Gradient Visualization**

Visualize LSTM gate mechanisms and compare gradient flow in vanilla RNNs vs LSTMs.

In [ ]:
!pip install -q torch matplotlib

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# LSTM Gates Visualization
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 10); ax.set_ylim(0, 8)
ax.axis('off')

# Cell body
cell_rect = mpatches.FancyBboxPatch((2, 2), 6, 4, boxstyle="round,pad=0.2",
                                     facecolor='#E8EAF6', edgecolor=MAIN_BLUE, lw=2)
ax.add_patch(cell_rect)

# Gates
gate_props = [
    (2.8, 3.5, 'Forget\nGate', IDA_RED),
    (4.3, 3.5, 'Input\nGate', FOREST),
    (5.8, 3.5, 'Cell\nUpdate', AMBER),
    (7.3, 3.5, 'Output\nGate', '#9C27B0'),
]

for x, y, label, color in gate_props:
    gate = mpatches.FancyBboxPatch((x-0.5, y-0.5), 1.0, 1.0,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='black',
                                    lw=1.5, alpha=0.3)
    ax.add_patch(gate)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

ax.text(5, 6.5, 'LSTM Cell', ha='center', fontsize=18, fontweight='bold', color=MAIN_BLUE)
ax.text(1, 4, '$h_{t-1}$', ha='center', fontsize=14, fontweight='bold')
ax.text(9, 4, '$h_t$', ha='center', fontsize=14, fontweight='bold')
ax.text(5, 1, '$x_t$', ha='center', fontsize=14, fontweight='bold')
ax.text(5, 7.2, '$c_t$ (cell state)', ha='center', fontsize=12, color='grey')

arrow_props = dict(arrowstyle='->', color=MAIN_BLUE, lw=2)
ax.annotate('', xy=(2, 4), xytext=(1.3, 4), arrowprops=arrow_props)
ax.annotate('', xy=(8.7, 4), xytext=(8, 4), arrowprops=arrow_props)
ax.annotate('', xy=(5, 2), xytext=(5, 1.3), arrowprops=arrow_props)

ax.annotate('', xy=(8.2, 6.8), xytext=(1.8, 6.8),
            arrowprops=dict(arrowstyle='->', color=FOREST, lw=3, ls='-'))
ax.text(5, 7.5, 'Cell state highway (gradient preservation)', ha='center',
        fontsize=11, color=FOREST, style='italic')

# Equations box
eq_text = ('$f_t = \\sigma(W_f [h_{t-1}, x_t] + b_f)$\n'
           '$i_t = \\sigma(W_i [h_{t-1}, x_t] + b_i)$\n'
           '$\\tilde{c}_t = \\tanh(W_c [h_{t-1}, x_t] + b_c)$\n'
           '$o_t = \\sigma(W_o [h_{t-1}, x_t] + b_o)$\n'
           '$c_t = f_t \\odot c_{t-1} + i_t \\odot \\tilde{c}_t$\n'
           '$h_t = o_t \\odot \\tanh(c_t)$')
ax.text(5, 0.6, eq_text.replace('\\n', '\n'),
        ha='center', va='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
save_fig('nn_ch05_lstm_gates')

In [ ]:
# Vanishing Gradient: RNN vs LSTM
import torch.nn as nn

seq_lengths = [10, 25, 50, 100]
hidden_size = 32

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vanilla RNN
ax = axes[0]
for seq_len in seq_lengths:
    torch.manual_seed(42)
    rnn = nn.RNN(1, hidden_size, batch_first=True)
    x = torch.randn(1, seq_len, 1, requires_grad=True)
    h = torch.zeros(1, 1, hidden_size)
    out, _ = rnn(x, h)

    grads = []
    for t in range(seq_len):
        if x.grad is not None:
            x.grad.zero_()
        out[0, t, 0].backward(retain_graph=True)
        grads.append(x.grad[0, 0, 0].item())

    ax.plot(range(seq_len), grads[::-1], lw=1.5, label=f'T={seq_len}', alpha=0.8)

ax.set_xlabel('Steps back from output')
ax.set_ylabel('Gradient magnitude')
ax.set_title('Vanilla RNN: Vanishing Gradients', fontweight='bold')
ax.legend(); ax.set_yscale('symlog', linthresh=1e-10)

# LSTM
ax = axes[1]
for seq_len in seq_lengths:
    torch.manual_seed(42)
    lstm = nn.LSTM(1, hidden_size, batch_first=True)
    x = torch.randn(1, seq_len, 1, requires_grad=True)
    out, _ = lstm(x)

    grads = []
    for t in range(seq_len):
        if x.grad is not None:
            x.grad.zero_()
        out[0, t, 0].backward(retain_graph=True)
        grads.append(x.grad[0, 0, 0].item())

    ax.plot(range(seq_len), grads[::-1], lw=1.5, label=f'T={seq_len}', alpha=0.8)

ax.set_xlabel('Steps back from output')
ax.set_ylabel('Gradient magnitude')
ax.set_title('LSTM: Gradient Preservation', fontweight='bold')
ax.legend(); ax.set_yscale('symlog', linthresh=1e-10)

plt.suptitle('Vanishing Gradient: RNN vs LSTM', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch05_vanishing_gradient')

## Summary

Visualized the **LSTM architecture** with its four gates and the cell state highway. Demonstrated how LSTMs mitigate the vanishing gradient problem compared to vanilla RNNs.